# Optimization Model Notebook

This notebook builds and solves a Vehicle-to-Grid (V2G) revenue maximization model using Mixed-Integer Linear Programming (MILP) with Google OR-Tools.

In [ ]:
from ortools.linear_solver import pywraplp
import numpy as np

solver = pywraplp.Solver.CreateSolver('SCIP')
if not solver:
    raise RuntimeError('SCIP solver was not created. Please verify OR-Tools installation.')

print('Solver initialized successfully.')

In [ ]:
# Sets
T = range(24)
EVS = range(4)
MARKETS = ['DA', 'ID']

# EV parameters
battery_capacity = {i: 45 + 5 * i for i in EVS}
initial_energy = {i: 0.60 * battery_capacity[i] for i in EVS}
min_energy = {i: 0.20 * battery_capacity[i] for i in EVS}
max_energy = {i: battery_capacity[i] for i in EVS}
max_charge = {i: 11.0 for i in EVS}
max_discharge = {i: 11.0 for i in EVS}

# Technical parameters
eta_ch = 0.93
eta_dch = 0.92
delta_t = 1.0

# Availability and trip consumption
availability = {(t, i): 1 if (0 <= t <= 7 or 18 <= t <= 23) else 0 for t in T for i in EVS}
trip_energy = {(t, i): 0.0 for t in T for i in EVS}
trip_energy[8, 0] = 5.0
trip_energy[8, 1] = 6.0
trip_energy[17, 2] = 4.0
trip_energy[17, 3] = 5.0

# Grid capacity
grid_cap = {t: 32.0 for t in T}

# Synthetic market prices
np.random.seed(7)
buy_price = {}
sell_price = {}
for t in T:
    for m in MARKETS:
        base = 0.11 if m == 'DA' else 0.13
        buy_price[t, m] = base + np.random.uniform(0.01, 0.05)
        sell_price[t, m] = buy_price[t, m] + np.random.uniform(0.02, 0.05)

print('Input data generated.')

In [ ]:
# Decision variables
p_buy = {(t, m): solver.NumVar(0, solver.infinity(), f'p_buy_{t}_{m}') for t in T for m in MARKETS}
p_sell = {(t, m): solver.NumVar(0, solver.infinity(), f'p_sell_{t}_{m}') for t in T for m in MARKETS}
p_ch = {(t, i): solver.NumVar(0, max_charge[i], f'p_ch_{t}_{i}') for t in T for i in EVS}
p_dch = {(t, i): solver.NumVar(0, max_discharge[i], f'p_dch_{t}_{i}') for t in T for i in EVS}
energy = {(t, i): solver.NumVar(min_energy[i], max_energy[i], f'energy_{t}_{i}') for t in T for i in EVS}
u_ch = {(t, i): solver.IntVar(0, 1, f'u_ch_{t}_{i}') for t in T for i in EVS}
u_dch = {(t, i): solver.IntVar(0, 1, f'u_dch_{t}_{i}') for t in T for i in EVS}

print('Decision variables created.')

In [ ]:
# Objective
revenue = solver.Sum(p_sell[t, m] * sell_price[t, m] * delta_t for t in T for m in MARKETS)
cost = solver.Sum(p_buy[t, m] * buy_price[t, m] * delta_t for t in T for m in MARKETS)
solver.Maximize(revenue - cost)

# Constraints
for i in EVS:
    solver.Add(energy[0, i] == initial_energy[i])

for t in T:
    for i in EVS:
        if t > 0:
            solver.Add(
                energy[t, i] == energy[t - 1, i]
                + eta_ch * p_ch[t, i] * delta_t
                - (1.0 / eta_dch) * p_dch[t, i] * delta_t
                - trip_energy[t, i]
            )

        solver.Add(p_ch[t, i] <= max_charge[i] * u_ch[t, i])
        solver.Add(p_dch[t, i] <= max_discharge[i] * u_dch[t, i])
        solver.Add(u_ch[t, i] + u_dch[t, i] <= 1)

        solver.Add(p_ch[t, i] <= max_charge[i] * availability[t, i])
        solver.Add(p_dch[t, i] <= max_discharge[i] * availability[t, i])
        solver.Add(u_ch[t, i] <= availability[t, i])
        solver.Add(u_dch[t, i] <= availability[t, i])

for t in T:
    solver.Add(solver.Sum(p_buy[t, m] for m in MARKETS) == solver.Sum(p_ch[t, i] for i in EVS))
    solver.Add(solver.Sum(p_sell[t, m] for m in MARKETS) == solver.Sum(p_dch[t, i] for i in EVS))
    solver.Add(solver.Sum(p_buy[t, m] for m in MARKETS) <= grid_cap[t])
    solver.Add(solver.Sum(p_sell[t, m] for m in MARKETS) <= grid_cap[t])

print('Objective and constraints defined.')

In [ ]:
status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print('Status: OPTIMAL')
elif status == pywraplp.Solver.FEASIBLE:
    print('Status: FEASIBLE')
else:
    print(f'Status code: {status}')

if status in (pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE):
    print(f'Objective value: {solver.Objective().Value():.4f} EUR')

In [ ]:
if status in (pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE):
    total_buy = sum(p_buy[t, m].solution_value() for t in T for m in MARKETS)
    total_sell = sum(p_sell[t, m].solution_value() for t in T for m in MARKETS)
    print(f'Total purchased energy: {total_buy:.2f} kWh')
    print(f'Total sold energy: {total_sell:.2f} kWh')

    for i in EVS:
        e0 = energy[0, i].solution_value()
        ef = energy[max(T), i].solution_value()
        print(f'EV {i}: initial SOC={e0:.2f} kWh, final SOC={ef:.2f} kWh')

## Notes
- This notebook is intentionally compact and runnable end-to-end.
- You can extend it with degradation costs, reserve markets, and scenario analysis as additional sections.